# Remote Deployment with Hetzner Cloud

This notebook demonstrates deploying a netrun pool server to a cloud VM and running
a network that offloads computation to it.

Steps:
1. Provision a Hetzner Cloud server using `hcloud`
2. Deploy the `app/` folder using netrun's `deploy()` API
3. Run a network with a remote pool on the provisioned server
4. Tear down the server

**Prerequisites:**
- `hcloud` CLI installed and authenticated (`hcloud context create`)
- An SSH key registered in your Hetzner Cloud project (`hcloud ssh-key list`)
- The corresponding private key available locally
- A `.env` file in this directory (copy `.env.example` and fill in your values)

## Configuration

In [5]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

# --- Loaded from .env (see .env.example) ---
HCLOUD_SSH_KEY_NAME = os.environ["HCLOUD_SSH_KEY_NAME"]
SSH_PRIVATE_KEY_PATH = os.environ["SSH_PRIVATE_KEY_PATH"]

# --- Server settings ---
SERVER_NAME = "netrun-demo"
SERVER_TYPE = "cpx22"          # 2 vCPU, 4 GB RAM
SERVER_IMAGE = "ubuntu-24.04"
SERVER_LOCATION = "fsn1"      # Falkenstein, DE

# --- Deployment settings ---
REMOTE_DIR = "/opt/netrun-app"
POOL_SERVER_PORT = 8765
APP_DIR = str(Path("./app").resolve())

## Provision Server

In [ ]:
import subprocess
import time
import socket

def run(cmd, **kwargs):
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{result.stderr}")
    return result

# Create the server
print(f"Creating server '{SERVER_NAME}' ({SERVER_TYPE}, {SERVER_IMAGE})...")
run([
    "hcloud", "server", "create",
    "--name", SERVER_NAME,
    "--type", SERVER_TYPE,
    "--image", SERVER_IMAGE,
    "--location", SERVER_LOCATION,
    "--ssh-key", HCLOUD_SSH_KEY_NAME,
])

# Get the server's IP
result = run(["hcloud", "server", "ip", SERVER_NAME])
SERVER_IP = result.stdout.strip()
print(f"Server IP: {SERVER_IP}")

# Wait for SSH
print("Waiting for SSH", end="", flush=True)
for _ in range(60):
    try:
        s = socket.create_connection((SERVER_IP, 22), timeout=2)
        s.close()
        print(" ready!")
        break
    except (ConnectionRefusedError, OSError, socket.timeout):
        print(".", end="", flush=True)
        time.sleep(2)
else:
    raise TimeoutError("SSH not ready after 120s")

# Give sshd a moment to fully start
time.sleep(3)

# Update known_hosts so pyinfra can connect without host key errors
# (Hetzner reuses IPs, so stale entries are common)
subprocess.run(["ssh-keygen", "-R", SERVER_IP], capture_output=True)
run(["ssh-keyscan", "-H", SERVER_IP])
keyscan = run(["ssh-keyscan", "-H", SERVER_IP])
known_hosts = Path.home() / ".ssh" / "known_hosts"
with open(known_hosts, "a") as f:
    f.write(keyscan.stdout)
print(f"Updated known_hosts for {SERVER_IP}")

## Deploy

Uploads the `app/` folder to the server, installs dependencies with `uv`, and
starts the pool server process in the background.

In [9]:
from netrun.deploy._config import (
    DeployConfig,
    SSHConfig,
    RepoConfig,
    UvEnvSetup,
    ConfigFileNetSource,
    PoolServerConfig,
)
from netrun.deploy._deploy import deploy

deploy_config = DeployConfig(
    ssh=SSHConfig(
        host=SERVER_IP,
        user="root",
        ssh_key=str(Path(SSH_PRIVATE_KEY_PATH).expanduser()),
    ),
    repo=RepoConfig(
        local_folder_path=APP_DIR,
        remote_dir=REMOTE_DIR,
    ),
    env_setup=UvEnvSetup(python_version="3.11"),
    net_source=ConfigFileNetSource(path="net_config.toml"),
    pool_server=PoolServerConfig(port=POOL_SERVER_PORT),
    pre_deploy_commands=[
        "apt-get update -qq && apt-get install -y -qq rsync > /dev/null",
    ],
)

print("Deploying...")
deploy_result = deploy(deploy_config)
assert deploy_result.success, f"Deploy failed: {deploy_result.errors}"
print(f"Pool server URL: {deploy_result.pool_server_url}")

Deploying...
Pool server URL: ws://188.245.171.241:8765


## Wait for Pool Server

In [10]:
print("Waiting for pool server", end="", flush=True)
for _ in range(60):
    try:
        s = socket.create_connection((SERVER_IP, POOL_SERVER_PORT), timeout=2)
        s.close()
        print(" ready!")
        break
    except (ConnectionRefusedError, OSError, socket.timeout):
        print(".", end="", flush=True)
        time.sleep(2)
else:
    raise TimeoutError("Pool server not ready after 120s")

Waiting for pool server................................. ready!


## Run the Network

The `find_primes` function runs on the remote server. We inject a range and
collect the results locally.

In [ ]:
import sys

# Add app/ to path so the client can resolve the function factory
sys.path.insert(0, APP_DIR)

from netrun.net import Net
from netrun.net.config._net_config import NetConfig, PoolConfig, RemotePoolConfig
from netrun.net.config._graph import GraphConfig
from netrun.net.config._nodes import NodeConfig, NodeExecutionConfig

client_config = NetConfig(
    pools={
        "remote": PoolConfig(
            spec=RemotePoolConfig(
                url=f"ws://{SERVER_IP}:{POOL_SERVER_PORT}",
                worker_name="execution_manager",
                num_processes=1,
                threads_per_process=1,
            ),
        ),
    },
    graph=GraphConfig(
        nodes=[
            NodeConfig(
                factory="netrun.node_factories.from_function",
                factory_args={"func": "nodes.find_primes"},
                execution_config=NodeExecutionConfig(pools=["remote"]),
            ),
        ],
    ),
)

async with Net(client_config) as net:
    net.inject_data("find_primes", "start", [0])
    net.inject_data("find_primes", "stop", [1000])

    await net.run_until_blocked()

    results = net.flush_all_output_queues()
    primes = [v for vals in results.values() for v in vals[0]]
    primes.sort()

    print(f"Found {len(primes)} primes in [0, 1000)")
    print(f"First 10: {primes[:10]}")
    print(f"Last 10:  {primes[-10:]}")

    await net.request_pool_shutdown("remote")

## Logs

In [ ]:
net.print_all_logs()

## Cleanup

Delete the Hetzner Cloud server. **Always run this cell** to avoid ongoing charges.

In [ ]:
print(f"Deleting server '{SERVER_NAME}'...")
subprocess.run(["hcloud", "server", "delete", SERVER_NAME], check=True)
print("Server deleted.")